# Phase I: Conditional VAE (Baseline Validation)

This notebook implements the **Phase I Conditional VAE** described in the ChronoDiff proposal. It serves two purposes:

1. **Baseline validation** — a sanity check that the slice-level data pipeline works and the model can reconstruct MRI slices cleanly.
2. **Latent encoder for Phase II** — the trained VAE encoder will be frozen and reused as the backbone for the latent-space Time-Conditioned Diffusion model in Phase II.

### Architecture
- **Input:** (1, 256, 256) MRI slice (resized from 192×192)
- **Encoder:** Conv2d(1→32, s=2) → Conv2d(32→64, s=2) → Conv2d(64→128, s=2)
- **Latent:** Linear(128·32·32 → 64) for μ and logσ² (separate heads)
- **Reparameterization:** z = μ + σ·ε
- **Decoder:** Linear(64 → 128·32·32) → ConvT(128→64, s=2) → ConvT(64→32, s=2) → ConvT(32→1, s=2, Sigmoid)
- **Output:** (1, 256, 256) reconstruction

### Conditioning
Phase I is **unconditional reconstruction** — we learn a good latent representation of brain slices. Phase II will add Δt conditioning to predict future slices from this latent space

## 1. Setup: Mount Drive & GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

Mon May 11 11:29:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q torch torchvision nibabel scikit-image lpips tqdm pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00


## 2. Imports & configuration

In [ ]:
import os, json, math, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [ ]:
PROJECT_DIR   = Path('/content/drive/MyDrive/JHU/GMCV/project')
MANIFEST_CSV  = PROJECT_DIR / 'manifests' / 'slices_manifest_full_100.csv'
OUT_DIR       = PROJECT_DIR / 'chronodiff' / 'PHASE1_VAE' / 'sample_100'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH     = OUT_DIR / 'vae_phase1.pt'
LOG_PATH      = OUT_DIR / 'train_log.json'

# --- Hyperparameters ---
IMG_SIZE      = 256       # resize 192→256 to match proposal architecture
BATCH_SIZE    = 64
N_EPOCHS      = 100
LR            = 2e-4
LATENT_DIM    = 64
BETA_KL       = 1e-5       # KL weight (β-VAE). Low β for reconstruction quality
NUM_WORKERS   = 8

print('Manifest:', MANIFEST_CSV)
print('Output:  ', OUT_DIR)
print('Checkpoint:', CKPT_PATH)

Manifest: /content/drive/MyDrive/JHU/GMCV/project/manifests/slices_manifest_full_100.csv
Output:   /content/drive/MyDrive/JHU/GMCV/project/chronodiff/PHASE1_VAE/sample_100
Checkpoint: /content/drive/MyDrive/JHU/GMCV/project/chronodiff/PHASE1_VAE/sample_100/vae_phase1.pt


## 3. Dataset

For Phase I we only need **individual slices** — not pairs. We reconstruct slices by themselves. So we take both `slice_t` and `slice_next` from the manifest and pool them together as single-slice training data.

Each slice is stored as a `.npy` file containing a 2D array, brain-masked and z-score normalized during preprocessing. We clip to a reasonable range and rescale to [0, 1] for the Sigmoid output.

In [ ]:
manifest = pd.read_csv(MANIFEST_CSV)
print('Manifest rows (pairs):', len(manifest))
print(manifest.columns.tolist())
manifest.head(3)

Manifest rows (pairs): 52907
['patient_id', 'pair_id', 'slice_id', 'z_idx', 'split', 'slice_t', 'slice_next', 'delta_t_days', 'brain_frac']


,patient_id,pair_id,slice_id,z_idx,split,slice_t,slice_next,delta_t_days,brain_frac
0,YG_05MQ79QMQ47J,YG_05MQ79QMQ47J_0_1,YG_05MQ79QMQ47J_0_1_z054,54,train,/content/drive/MyDrive/project/dataset/full_sl...,/content/drive/MyDrive/project/dataset/full_sl...,18,NaN
1,YG_05MQ79QMQ47J,YG_05MQ79QMQ47J_0_1,YG_05MQ79QMQ47J_0_1_z057,57,train,/content/drive/MyDrive/project/dataset/full_sl...,/content/drive/MyDrive/project/dataset/full_sl...,18,NaN
2,YG_05MQ79QMQ47J,YG_05MQ79QMQ47J_0_1,YG_05MQ79QMQ47J_0_1_z060,60,train,/content/drive/MyDrive/project/dataset/full_sl...,/content/drive/MyDrive/project/dataset/full_sl...,18,NaN


In [ ]:
# Define the old and new path prefixes
old_path = '/content/drive/MyDrive/project/dataset/'
new_path = '/content/drive/MyDrive/JHU/GMCV/project/dataset/'

# Update the 'slice_t' column
manifest['slice_t'] = manifest['slice_t'].str.replace(old_path, new_path, regex=False)

# Update the 'slice_next' column
manifest['slice_next'] = manifest['slice_next'].str.replace(old_path, new_path, regex=False)

manifest.head(3)

,patient_id,pair_id,slice_id,z_idx,split,slice_t,slice_next,delta_t_days,brain_frac
0,YG_05MQ79QMQ47J,YG_05MQ79QMQ47J_0_1,YG_05MQ79QMQ47J_0_1_z054,54,train,/content/drive/MyDrive/JHU/GMCV/project/datase...,/content/drive/MyDrive/JHU/GMCV/project/datase...,18,NaN
1,YG_05MQ79QMQ47J,YG_05MQ79QMQ47J_0_1,YG_05MQ79QMQ47J_0_1_z057,57,train,/content/drive/MyDrive/JHU/GMCV/project/datase...,/content/drive/MyDrive/JHU/GMCV/project/datase...,18,NaN
2,YG_05MQ79QMQ47J,YG_05MQ79QMQ47J_0_1,YG_05MQ79QMQ47J_0_1_z060,60,train,/content/drive/MyDrive/JHU/GMCV/project/datase...,/content/drive/MyDrive/JHU/GMCV/project/datase...,18,NaN


In [ ]:
print(manifest['slice_t'].iloc[0])

/content/drive/MyDrive/JHU/GMCV/project/dataset/full_slices/YG_05MQ79QMQ47J/YG_05MQ79QMQ47J_0_1_z054_t.npy


In [ ]:
len(np.unique(manifest['patient_id']))

100

In [ ]:
# Patient-level 70/15/15 split
patients = sorted(manifest['patient_id'].unique())
rng = np.random.default_rng(SEED)
rng.shuffle(patients)
n = len(patients)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
train_p = set(patients[:n_train])
val_p   = set(patients[n_train:n_train+n_val])
test_p  = set(patients[n_train+n_val:])

def split_df(df, pset):
    return df[df['patient_id'].isin(pset)].reset_index(drop=True)

train_df = split_df(manifest, train_p)
val_df   = split_df(manifest, val_p)
test_df  = split_df(manifest, test_p)
print(f'Patients: train={len(train_p)}, val={len(val_p)}, test={len(test_p)}')
print(f'Pairs:    train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')

Patients: train=70, val=15, test=15
Pairs:    train=35398, val=12826, test=4683


In [ ]:
def pool_slices(df):
    """Concat slice_t and slice_next into a flat list of unique slice paths."""
    a = df['slice_t'].tolist()
    b = df['slice_next'].tolist()
    return sorted(set(a + b))

train_slices = pool_slices(train_df)
val_slices   = pool_slices(val_df)
test_slices  = pool_slices(test_df)
print(f'Unique slices — train: {len(train_slices)}, val: {len(val_slices)}, test: {len(test_slices)}')

Unique slices — train: 70796, val: 25652, test: 9366


In [ ]:
import os

def filter_valid_paths(path_list, split_name="Dataset"):
    valid_paths = []
    missing_paths = []

    for p in path_list:
        if os.path.isfile(p):
            valid_paths.append(p)
        else:
            missing_paths.append(p)

    print(f"[{split_name}] Total: {len(path_list)} | Valid: {len(valid_paths)} | Missing: {len(missing_paths)}")

    # Print an example of a missing path to help you debug why it's missing
    if missing_paths:
        print(f"   -> Example missing path: {missing_paths[0]}")

    return valid_paths

# Run the checkup and overwrite your lists with ONLY the valid paths
print("--- Running File Checkup ---")
train_slices_valid = filter_valid_paths(train_slices, "Train")
val_slices_valid   = filter_valid_paths(val_slices, "Val")
test_slices_valid  = filter_valid_paths(test_slices, "Test")
print("----------------------------\n")

--- Running File Checkup ---
[Train] Total: 70796 | Valid: 12376 | Missing: 58420
   -> Example missing path: /content/drive/MyDrive/JHU/GMCV/project/dataset/full_slices/YG_0L8F6DK4OHSS/YG_0L8F6DK4OHSS_0_1_z066_next.npy
[Val] Total: 25652 | Valid: 6602 | Missing: 19050
   -> Example missing path: /content/drive/MyDrive/JHU/GMCV/project/dataset/full_slices/YG_1GVBTE7H79DA/YG_1GVBTE7H79DA_0_3_z039_t.npy
[Test] Total: 9366 | Valid: 5693 | Missing: 3673
   -> Example missing path: /content/drive/MyDrive/JHU/GMCV/project/dataset/full_slices/YG_2LEC0G5PJYWI/YG_2LEC0G5PJYWI_0_1_z075_next.npy
----------------------------



In [ ]:
import random
import numpy as np
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader


class SliceDataset(Dataset):
    def __init__(self, slice_paths, img_size=256, augment=False):
        self.paths = slice_paths
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def _load(self, p):
        arr = np.load(p).astype(np.float32)           # z-scored, brain-masked
        arr = np.clip(arr, -3.0, 3.0)                 # clip outliers
        arr = (arr + 3.0) / 6.0                       # -> [0, 1]
        t = torch.from_numpy(arr).unsqueeze(0)        # (1, H, W)
        if t.shape[-1] != self.img_size:
            t = TF.resize(t, [self.img_size, self.img_size], antialias=True)
        return t

    def _augment(self, x):
        if random.random() < 0.5:
            x = torch.flip(x, dims=[-1])

        if random.random() < 0.5:
            angle = random.uniform(-15.0, 15.0)
            x = TF.rotate(x, angle, interpolation=TF.InterpolationMode.BILINEAR, fill=0.5)

        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.0)
            crop  = int(self.img_size * scale)
            top   = random.randint(0, self.img_size - crop)
            left  = random.randint(0, self.img_size - crop)
            x = TF.resized_crop(x, top, left, crop, crop,
                                size=[self.img_size, self.img_size],
                                antialias=True)

        if random.random() < 0.3:
            contrast   = random.uniform(0.9, 1.1)
            brightness = random.uniform(-0.05, 0.05)
            x = (x - 0.5) * contrast + 0.5 + brightness
            x = x.clamp(0.0, 1.0)

        if random.random() < 0.25:
            eh = random.randint(16, 32)
            ew = random.randint(16, 32)
            ey = random.randint(0, self.img_size - eh)
            ex = random.randint(0, self.img_size - ew)
            x[..., ey:ey+eh, ex:ex+ew] = 0.5   # fill with background value

        return x

    def __getitem__(self, i):
        x = self._load(self.paths[i])
        if self.augment:
            x = self._augment(x)
        return x

train_ds = SliceDataset(train_slices, IMG_SIZE, augment=True)
val_ds   = SliceDataset(val_slices,   IMG_SIZE, augment=False)
test_ds  = SliceDataset(test_slices,  IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Sanity check
xb = next(iter(train_loader))
print('Batch shape:', xb.shape, 'range:', xb.min().item(), xb.max().item())

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1712/1486550482.py", line 87, in __getitem__
    x = self._load(self.paths[i])
        ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1712/1486550482.py", line 39, in _load
    arr = np.load(p).astype(np.float32)           # z-scored, brain-masked
          ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 455, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                            

## 4. Conditional VAE model (matches proposal slide)

Architecture:
- **Encoder**: three stride-2 Conv2d layers, 1 → 32 → 64 → 128 channels, with ReLU. Input 256×256 → output 128×32×32.
- **Latent**: two `Linear(128·32·32 → 64)` heads for μ and logσ².
- **Reparameterization**: z = μ + σ·ε, ε ~ N(0, I).
- **Decoder**: `Linear(64 → 128·32·32)` reshape → three stride-2 ConvTranspose2d layers, 128 → 64 → 32 → 1, Sigmoid on output.

In [ ]:
class ConditionalVAE(nn.Module):
    """
    Spatial-latent VAE. Encoder produces a (B, latent_ch, 32, 32) Gaussian.
    Total latent dim: latent_ch * 32 * 32 = 4 * 32 * 32 = 4,096 (vs 64 before).
    """
    def __init__(self, latent_ch: int = 4, base_ch: int = 32):
        super().__init__()
        self.latent_ch = latent_ch

        self.enc = nn.Sequential(
            nn.Conv2d(1,          base_ch,     4, stride=2, padding=1),  # 256→128
            nn.GroupNorm(8, base_ch),
            nn.SiLU(),
            nn.Conv2d(base_ch,    base_ch * 2, 4, stride=2, padding=1),  # 128→64
            nn.GroupNorm(8, base_ch * 2),
            nn.SiLU(),
            nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1), # 64→32
            nn.GroupNorm(8, base_ch * 4),
            nn.SiLU(),
        )
        self.to_latent = nn.Conv2d(base_ch * 4, 2 * latent_ch, kernel_size=1)

        self.from_latent = nn.Conv2d(latent_ch, base_ch * 4, kernel_size=1)
        self.dec = nn.Sequential(
            nn.SiLU(),
            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, stride=2, padding=1),  # 32→64
            nn.GroupNorm(8, base_ch * 2),
            nn.SiLU(),
            nn.ConvTranspose2d(base_ch * 2, base_ch,     4, stride=2, padding=1),  # 64→128
            nn.GroupNorm(8, base_ch),
            nn.SiLU(),
            nn.ConvTranspose2d(base_ch,     1,           4, stride=2, padding=1),  # 128→256
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x)
        h = self.to_latent(h)                       # (B, 2*latent_ch, 32, 32)
        mu, logvar = h.chunk(2, dim=1)              # each (B, latent_ch, 32, 32)
        return mu, logvar

    def reparam(self, mu, logvar):
        std = (0.5 * logvar).exp()
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.dec(self.from_latent(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar


model = ConditionalVAE(latent_ch=LATENT_DIM, base_ch=32).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'VAE params: {n_params/1e6:.2f} M')
# Shape sanity check
with torch.no_grad():
    xb = xb.to(DEVICE)
    x_hat, mu, logvar = model(xb)
    print('x_hat:', x_hat.shape, 'mu:', mu.shape, 'logvar:', logvar.shape)

## 5. Loss: reconstruction + KL divergence

Standard β-VAE loss:

$$\mathcal{L} = \underbrace{\|x - \hat{x}\|_2^2}_{\text{MSE reconstruction}} + \beta \cdot \underbrace{D_{KL}(q(z|x) \,\|\, \mathcal{N}(0, I))}_{\text{KL regularizer}}$$

We use MSE (not BCE) because MRI slice intensities are continuous. We average per pixel so the scale matches KL.
KL is in closed form for a Gaussian prior: $-\tfrac12 \sum (1 + \log\sigma^2 - \mu^2 - \sigma^2)$.

In [ ]:
def vae_loss(x, x_hat, mu, logvar, beta=1e-3):
    recon = F.mse_loss(x_hat, x, reduction='mean')                           # per-pixel MSE
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())          # per-latent-dim KL
    loss  = recon + beta * kl
    return loss, recon.detach(), kl.detach()

## 6. Training loop

In [ ]:
import warnings, logging
warnings.filterwarnings('ignore', message='.*child process.*')
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

history = {'epoch': [], 'train_loss': [], 'train_recon': [], 'train_kl': [],
           'val_loss': [], 'val_recon': [], 'val_kl': [], 'lr': []}
best_val = float('inf')

def run_epoch(loader, train=True):
    model.train(train)
    totals = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0, 'n': 0}
    pbar = tqdm(loader, desc='train' if train else 'val', leave=False)
    for x in pbar:
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            x_hat, mu, logvar = model(x)
            loss, recon, kl = vae_loss(x, x_hat, mu, logvar, beta=BETA_KL)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        bs = x.size(0)
        totals['loss']  += loss.item()  * bs
        totals['recon'] += recon.item() * bs
        totals['kl']    += kl.item()    * bs
        totals['n']     += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}", recon=f"{recon.item():.4f}", kl=f"{kl.item():.4f}")
    return {k: v / max(totals['n'], 1) for k, v in totals.items() if k != 'n'}

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va = run_epoch(val_loader, train=False)
    sched.step()
    lr_now = opt.param_groups[0]['lr']
    dt = time.time() - t0
    print(f'Epoch {epoch:02d}/{N_EPOCHS} ({dt:.0f}s) | '
          f'train loss {tr["loss"]:.4f} (recon {tr["recon"]:.4f}, kl {tr["kl"]:.4f}) | '
          f'val loss {va["loss"]:.4f} (recon {va["recon"]:.4f}, kl {va["kl"]:.4f}) | lr {lr_now:.2e}')
    history['epoch'].append(epoch)
    history['train_loss'].append(tr['loss']); history['train_recon'].append(tr['recon']); history['train_kl'].append(tr['kl'])
    history['val_loss'].append(va['loss']);   history['val_recon'].append(va['recon']);   history['val_kl'].append(va['kl'])
    history['lr'].append(lr_now)
    if va['loss'] < best_val - 1e-5:
        best_val = va['los']
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_loss': best_val,
                    'config': {'latent_dim': LATENT_DIM, 'img_size': IMG_SIZE, 'beta': BETA_KL}},
                   CKPT_PATH)
        print(f'  ↳ saved best checkpoint (val={best_val:.4f}) to {CKPT_PATH}')
    with open(LOG_PATH, 'w') as f:
        json.dump(history, f, indent=2)

## 7. Training curves

In [ ]:
# load json file
with open(LOG_PATH, 'r') as f:
    history = json.load(f)

# get train_loss, val_loss, train_recon, val_recon, train_kl, val_kl
history = {k: np.array(v) for k, v in history.items()}

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(history['epoch'], history['train_loss'], label='train')
ax[0].plot(history['epoch'], history['val_loss'],   label='val')
ax[0].set_title('Total loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history['epoch'], history['train_recon'], label='train')
ax[1].plot(history['epoch'], history['val_recon'],   label='val')
ax[1].set_title('Reconstruction (MSE)'); ax[1].set_xlabel('epoch'); ax[1].legend()
ax[2].plot(history['epoch'], history['train_kl'], label='train')
ax[2].plot(history['epoch'], history['val_kl'],   label='val')
ax[2].set_title('KL divergence'); ax[2].set_xlabel('epoch'); ax[2].legend()
plt.tight_layout(); plt.savefig(OUT_DIR/'training_curves.png', dpi=120); plt.show()

## 8. Qualitative validation: reconstructions

A working VAE should reconstruct input slices with minor blur but correct anatomy.

In [ ]:
# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model']); model.eval()
print('Loaded epoch', ckpt['epoch'], 'val', ckpt['val_loss'])

with torch.no_grad():
    x = next(iter(val_loader)).to(DEVICE)[:8]
    x_hat, mu, logvar = model(x)
    x, x_hat = x.cpu(), x_hat.cpu()

fig, ax = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    ax[0, i].imshow(x[i, 0], cmap='gray'); ax[0, i].axis('off')
    ax[1, i].imshow(x_hat[i, 0], cmap='gray'); ax[1, i].axis('off')
ax[0, 0].set_title('Input', loc='left'); ax[1, 0].set_title('Reconstruction', loc='left')
plt.tight_layout(); plt.savefig(OUT_DIR/'reconstructions.png', dpi=120); plt.show()

## 9. Latent space samples

Draw samples from the prior N(0, I) and decode. These will look smooth/averaged since Phase I is a small VAE — this is expected. Phase II (diffusion in latent space) will produce sharper images.

In [ ]:
with torch.no_grad():
    z = torch.randn(8, LATENT_DIM, 32, 32, device=DEVICE)
    samples = model.decode(z).cpu()

fig, ax = plt.subplots(1, 8, figsize=(20, 3))
for i in range(8):
    ax[i].imshow(samples[i, 0], cmap='gray')
    ax[i].axis('off')

plt.suptitle(f'Samples from Spatial Prior N(0, I) - Shape: {z.shape[1:]}')
plt.tight_layout()
plt.show()

## 10. Quantitative test metrics

Reconstruction PSNR / SSIM on the held-out test set. These establish the Phase I baseline numbers you can report in the presentation.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

mae_l, mse_l, psnr_l, ssim_l = [], [], [], []
model.eval()
with torch.no_grad():
    for x in tqdm(test_loader, desc='test'):
        x = x.to(DEVICE)
        x_hat, _, _ = model(x)
        x_np     = x.cpu().numpy()
        x_hat_np = x_hat.cpu().numpy()
        for i in range(x_np.shape[0]):
            a = x_np[i, 0]; b = x_hat_np[i, 0]
            mae_l.append(float(np.mean(np.abs(a - b))))
            mse_l.append(float(np.mean((a - b) ** 2)))
            psnr_l.append(float(psnr_fn(a, b, data_range=1.0)))
            ssim_l.append(float(ssim_fn(a, b, data_range=1.0)))

metrics = {
    'MAE':  (float(np.mean(mae_l)),  float(np.std(mae_l))),
    'MSE':  (float(np.mean(mse_l)),  float(np.std(mse_l))),
    'PSNR': (float(np.mean(psnr_l)), float(np.std(psnr_l))),
    'SSIM': (float(np.mean(ssim_l)), float(np.std(ssim_l))),
}
print('Phase I VAE — Test reconstruction metrics')
for k, (m, s) in metrics.items():
    print(f'  {k:4s}: {m:.4f} ± {s:.4f}')
with open(OUT_DIR/'test_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)